In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
sns.set_theme(style="whitegrid", context="talk")


## Clean GDP per capita data


In [2]:
gdp_raw = pd.read_csv("../data/gdp_per_capita/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv", skiprows=4, encoding="utf-8-sig")
gdp_raw.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30645.890602,22759.807175,26749.329609,30975.998912,35718.753119,39498.594129,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089204,186.909053,197.367547,225.400079,208.962717,226.836135,...,1528.104224,1552.073722,1507.085600,1351.591669,1562.416175,1679.327622,1571.449189,1615.396356,NaN,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,NaN,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.936832,127.451040,133.823783,139.004980,148.545883,155.561897,...,1574.230564,1720.140092,2216.385055,2030.861659,2112.794076,2138.473153,1841.855064,1411.337029,NaN,NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2790.718869,2860.093648,2493.678844,1759.356199,2303.908127,3682.113151,2916.136633,2665.874448,NaN,NaN


In [3]:
year_cols = [c for c in gdp_raw.columns if c.isdigit()]

gdp_long = gdp_raw[["Country Name", "Country Code", *year_cols]].melt(
    id_vars=["Country Name", "Country Code"],
    var_name="year",
    value_name="gdp_usd"
)

gdp_long["Country Code"] = gdp_long["Country Code"].str.upper().str.strip()
gdp_long["year"] = pd.to_numeric(gdp_long["year"], errors="coerce").astype("Int64")
gdp_long["gdp_usd"] = pd.to_numeric(gdp_long["gdp_usd"], errors="coerce")

gdp_long = gdp_long.sort_values(["Country Code", "year"]).reset_index(drop=True)

# Optional fallback: fill missing GDP by interpolation within each country time series
gdp_long["gdp_usd_filled"] = (
    gdp_long.groupby("Country Code")["gdp_usd"]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)

gdp_long.head()


,Country Name,Country Code,year,gdp_usd,gdp_usd_filled
0,Aruba,ABW,1960,NaN,6767.559229
1,Aruba,ABW,1961,NaN,6767.559229
2,Aruba,ABW,1962,NaN,6767.559229
3,Aruba,ABW,1963,NaN,6767.559229
4,Aruba,ABW,1964,NaN,6767.559229


In [4]:
olympic_df = pd.read_csv("../data/olympic_medals.csv")

print(olympic_df.shape)
olympic_df.head()


(21261, 9)


,season,year,medal,country_code,country,games,sport,event_gender,event_name
0,Summer,1896,Gold,USA,United States,1896 Athens,Athletics,Men's,100m
1,Summer,1896,Silver,GER,Germany,1896 Athens,Athletics,Men's,100m
2,Summer,1896,Bronze,HUN,Hungary,1896 Athens,Athletics,Men's,100m
3,Summer,1896,Bronze,USA,United States,1896 Athens,Athletics,Men's,100m
4,Summer,1896,Gold,USA,United States,1896 Athens,Athletics,Men's,400m


In [5]:
# clean keys
olympic_df["country_code"] = olympic_df["country_code"].str.upper().str.strip()
olympic_df["year"] = pd.to_numeric(olympic_df["year"], errors="coerce").astype("Int64")



In [6]:
olympic_modern_df = olympic_df[olympic_df["year"] >= 1960].copy()

merged = olympic_modern_df.merge(
    gdp_long[["Country Code", "year", "gdp_usd"]],
    left_on=["country_code", "year"],
    right_on=["Country Code", "year"],
    how="left"
).drop(columns=["Country Code"])

print("GDP filled rate (before):", merged["gdp_usd"].notna().mean())


missing = merged.loc[merged["gdp_usd"].isna(), ["country_code", "year"]].drop_duplicates()
gdp_codes = set(gdp_long["Country Code"])

missing["reason"] = np.where(
    missing["country_code"].isin(gdp_codes),
    "code_exists_but_year_missing",
    "code_not_in_gdp"
)

print(missing["reason"].value_counts())
print("\nTop codes not in GDP:")
print(
    missing.loc[missing["reason"] == "code_not_in_gdp", "country_code"]
    .value_counts()
)


GDP filled rate (before): 0.6870752464336007
code_not_in_gdp                 385
code_exists_but_year_missing     38
Name: reason, dtype: int64

Top codes not in GDP:
NED    25
SUI    25
BUL    19
DEN    18
GER    17
       ..
PAR     1
EUN     1
IOP     1
SRI     1
BUR     1
Name: country_code, Length: 62, dtype: int64


In [7]:
code_fix = {
    # IOC -> WB/ISO3 mappings
    "SUI": "CHE",  # Switzerland
    "NED": "NLD",  # Netherlands
    "BUL": "BGR",  # Bulgaria
    "DEN": "DNK",  # Denmark
    "GER": "DEU",  # Germany
    "SLO": "SVN",  # Slovenia
    "GRE": "GRC",  # Greece
    "IRI": "IRN",  # Iran
    "MGL": "MNG",  # Mongolia
    "CRO": "HRV",  # Croatia
    "POR": "PRT",  # Portugal
    "LAT": "LVA",  # Latvia
    "NGR": "NGA",  # Nigeria
    "RSA": "ZAF",  # South Africa
    "INA": "IDN",  # Indonesia
    "BAH": "BHS",  # Bahamas
    "ALG": "DZA",  # Algeria
    "MAS": "MYS",  # Malaysia
    "PHI": "PHL",  # Philippines
    "VIE": "VNM",  # Vietnam
    "BOT": "BWA",  # Botswana
    "CRC": "CRI",  # Costa Rica
    "CHI": "CHL",  # Chile
    "GRN": "GRD",  # Grenada
    "PUR": "PRI",  # Puerto Rico
    "URU": "URY",  # Uruguay
    "KOS": "XKX",  # Kosovo
    "KSA": "SAU",  # Saudi Arabia
    "ZIM": "ZWE",  # Zimbabwe
    "FIJ": "FJI",  # Fiji
    "ZAM": "ZMB",  # Zambia
    "KUW": "KWT",  # Kuwait
    "UAE": "ARE",  # United Arab Emirates
    "BER": "BMU",  # Bermuda
    "OAR": "RUS",  # Olympic Athletes from Russia
    "SCG": "SRB",  # Serbia and Montenegro (approx.)

    # Historical IOC codes mapped approximately to modern Germany
    "FRG": "DEU",  # West Germany
    "GDR": "DEU",  # East Germany
    "EUA": "DEU",  # Unified Team of Germany

    # Note: TPE has no GDP row in this WB file (no TWN code here)
    # Note: TCH, URS, YUG are historical entities and remain unmatched
}


In [8]:
olympic_modern_df["country_code_fix"] = olympic_modern_df["country_code"].replace(code_fix)

merged_fixed = olympic_modern_df.merge(
    gdp_long[["Country Code", "year", "gdp_usd", "gdp_usd_filled"]],
    left_on=["country_code_fix", "year"],
    right_on=["Country Code", "year"],
    how="left"
).drop(columns=["Country Code"])

# strict value from the exact year; fallback value from interpolated country series
merged_fixed["gdp_usd_final"] = merged_fixed["gdp_usd"].fillna(merged_fixed["gdp_usd_filled"])

print("GDP filled rate (strict):", merged_fixed["gdp_usd"].notna().mean())
print("GDP filled rate (with interpolation):", merged_fixed["gdp_usd_final"].notna().mean())

missing_final = merged_fixed.loc[
    merged_fixed["gdp_usd_final"].isna(),
    ["country_code", "country_code_fix", "year"],
].drop_duplicates()

gdp_codes = set(gdp_long["Country Code"])

missing_final["reason"] = np.where(
    missing_final["country_code_fix"].isin(gdp_codes),
    "code_exists_but_year_missing",
    "code_not_in_gdp"
)

print()
print("Missing breakdown (after fix + interpolation):")
print(missing_final["reason"].value_counts())

print()
print("Top original IOC codes still missing:")
print(missing_final["country_code"].value_counts().head(20))

print()
print("Top mapped codes still missing:")
print(missing_final["country_code_fix"].value_counts().head(20))

print()
print("Sample missing rows:")
print(missing_final.head(20))


GDP filled rate (strict): 0.8822016775852568
GDP filled rate (with interpolation): 0.9103655176636258

Missing breakdown (after fix + interpolation):
code_not_in_gdp                 59
code_exists_but_year_missing    12
Name: reason, dtype: int64

Top original IOC codes still missing:
TPE    12
PRK    12
TCH     9
YUG     8
URS     8
NIG     2
GUA     2
BWI     1
SUD     1
AIN     1
EOR     1
IOA     1
SAM     1
TOG     1
MRI     1
TGA     1
PAR     1
SRI     1
BAR     1
IOP     1
Name: country_code, dtype: int64

Top mapped codes still missing:
TPE    12
PRK    12
TCH     9
YUG     8
URS     8
NIG     2
GUA     2
BWI     1
SUD     1
AIN     1
EOR     1
IOA     1
SAM     1
TOG     1
MRI     1
TGA     1
PAR     1
SRI     1
BAR     1
IOP     1
Name: country_code_fix, dtype: int64

Sample missing rows:
     country_code country_code_fix  year                        reason
11            BWI              BWI  1960               code_not_in_gdp
18            URS              URS  1960       